# UVJ Template Selector

A small utility for picking representative Brown 2014 (GALSEDATLAS) galaxy
templates for the `student_edition/` walkthrough - or for any other
purpose that needs a galaxy sitting at a specific, deliberately-chosen spot
on the UVJ diagram.

**Why this exists**: an earlier pick for the "dusty" example turned out, on
inspection, to actually be classified *Quiescent* at $\alpha=0$ (no AGN) -
it looked plausible by name but was sitting right on top of a
classification boundary rather than clearly inside its intended region.
This notebook computes the real rest-frame UVJ position of every template
in the full atlas directly from its spectrum, so a choice can be checked
before it's used rather than assumed.

Run top to bottom, then use the ranked tables near the bottom to pick a
template with whatever boundary distance you want - "deep in a region" or
"close to a specific boundary" are both just a matter of which row you
pick.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.path as mpath

from src import config
from glass import data_io, photometry, visualization

plt.style.use('default')
visualization.apply_pasa_style()

## Load the full atlas and compute rest-frame UVJ for every template

No AGN is involved anywhere in this notebook - every point below is a
template's own spectrum, exactly as it comes out of the atlas
($\alpha=0$ in the walkthrough's language).

In [ ]:
brown_dir = os.path.join(config.RAW_DATA_DIR, 'Templates', 'Brown', '2014', 'Rest')
templates, names = data_io.read_brown_galaxy_templates(brown_dir)
print(f"Loaded {len(names)} templates from {brown_dir}")

filters = photometry.load_passbands(config.FILTER_PATHS)

rows = []
for name, template in zip(names, templates):
    uv, vj = photometry.calculate_UVJ_colours(template, filters['U'], filters['V'], filters['J'])
    if np.isnan(uv) or np.isnan(vj):
        continue
    rows.append({'name': name, 'VJ': vj, 'UV': uv})

df = pd.DataFrame(rows)
print(f"{len(df)} templates have valid UVJ colours")
df.head()

## Classify and measure distance from each boundary

Uses the *exact* quiescent-selection polygon and dusty/star-forming divide
from `photometry.classify_uvj`, so this matches the walkthrough notebook's
own classification rule exactly - no separate definition to drift out of
sync.

Three margins are computed for every template, regardless of which region
it currently falls in:

- `margin_quiescent`: signed vertical distance from the quiescent wedge's
  lower boundary (positive = inside/above the boundary, i.e. more deeply
  quiescent; negative = below it, in dusty/star-forming territory).
- `margin_vj12`: signed horizontal distance from the $V-J=1.2$ dusty/
  star-forming divide (positive = dusty side, negative = star-forming side).

Both are the natural "how far from crossing *this specific* boundary"
numbers - e.g. a quiescent galaxy with a small **positive**
`margin_quiescent` is sitting close to the star-forming/dusty side and
would flip classification with only a little added AGN light.

In [ ]:
def quiescent_lower_uv(vj):
    # Height of the quiescent wedge's lower boundary at a given V-J
    if vj <= 0.85:
        return 1.3
    elif vj <= 1.6:
        return 1.3 + (vj - 0.85) / (1.6 - 0.85) * (1.95 - 1.3)
    else:
        return 1.95

quiescent_verts = [(-0.5, 1.3), (0.85, 1.3), (1.6, 1.95), (1.6, 2.5), (-0.5, 2.5), (-0.5, 1.3)]
quiescent_path = mpath.Path(quiescent_verts)

def classify_and_measure(vj, uv):
    is_quiescent = quiescent_path.contains_point((vj, uv))
    region = 'Quiescent' if is_quiescent else ('Dusty' if vj > 1.2 else 'Star-forming')
    margin_quiescent = uv - quiescent_lower_uv(vj)
    margin_vj12 = vj - 1.2
    return region, margin_quiescent, margin_vj12

df[['region', 'margin_quiescent', 'margin_vj12']] = df.apply(
    lambda r: pd.Series(classify_and_measure(r['VJ'], r['UV'])), axis=1
)
df['region'].value_counts()

## The big picture: every template on the UVJ diagram

Colour-coded by region, using the project's own `visualization.plot_uvj_diagram`
(the exact same wedge, boundary line, and axis convention used everywhere
else in this project).

In [ ]:
region_to_code = {'Quiescent': 0, 'Star-forming': 1, 'Dusty': 2}
classifications = df['region'].map(region_to_code).values

fig, ax = plt.subplots(figsize=(7, 7))
visualization.plot_uvj_diagram(df['VJ'].values, df['UV'].values, classifications=classifications,
                                ax=ax, title=f'All {len(df)} Brown 2014 templates, rest-frame ($\\alpha$=0)')
plt.show()

## Zoomed view: who's near the quiescent / star-forming boundary?

This is the region that matters for "a quiescent galaxy that starts close
to the star-forming side." Labelled points are every **Quiescent**
template within 0.3 mag of the boundary (`margin_quiescent < 0.3`) plus
every **Star-forming** template within 0.3 mag on the other side, so you
can see exactly who's nearby and by how much.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))

quiescent_verts = [(-0.5, 1.3), (0.85, 1.3), (1.6, 1.95), (1.6, 2.5), (-0.5, 2.5), (-0.5, 1.3)]
ax.add_patch(plt.Polygon(quiescent_verts, closed=True, fill=True, facecolor='red',
                         alpha=0.08, edgecolor='#333333', lw=2.0, zorder=1))
ax.plot([1.2, 1.2], [0, 1.6], color='#333333', linestyle='--', lw=1.5, zorder=1)

near_boundary = df[(df['region'].isin(['Quiescent', 'Star-forming'])) & (df['margin_quiescent'].abs() < 0.3)]
other = df[~df.index.isin(near_boundary.index)]

ax.scatter(other['VJ'], other['UV'], c='lightgray', s=15, zorder=2)
colors = {'Quiescent': '#CC2929', 'Star-forming': '#1A6FB5', 'Dusty': '#E07B00'}
for region, sub in near_boundary.groupby('region'):
    ax.scatter(sub['VJ'], sub['UV'], c=colors[region], s=40, zorder=3, label=region, edgecolor='k', linewidth=0.4)
    for _, r in sub.iterrows():
        ax.annotate(r['name'], (r['VJ'], r['UV']), fontsize=7, xytext=(5, 3), textcoords='offset points')

ax.set_xlim(-0.5, 1.8)
ax.set_ylim(0.5, 2.0)
ax.set_xlabel('V - J (rest-frame)')
ax.set_ylabel('U - V (rest-frame)')
ax.set_title('Templates near the quiescent / star-forming boundary')
ax.legend(loc='upper left')
ax.grid(alpha=0.25)
plt.show()

## Ranked tables

Sorted so the *first* row of each table is always the closest to a
boundary - pick further down a table for a "deeper," more unambiguous
example, or from the top for one that sits close to crossing.

In [ ]:
print("=== Quiescent templates, ranked closest-to-star-forming-boundary first ===")
quiescent_ranked = df[df['region'] == 'Quiescent'].sort_values('margin_quiescent')
quiescent_ranked[['name', 'VJ', 'UV', 'margin_quiescent']].head(15)

In [ ]:
print("=== Star-forming templates, ranked closest-to-boundary first ===")
# margin_vj12 = VJ - 1.2, which is negative for star-forming galaxies;
# closest-to-boundary means closest to zero, i.e. largest (least negative) first.
sf_ranked = df[df['region'] == 'Star-forming'].sort_values('margin_vj12', ascending=False)
sf_ranked[['name', 'VJ', 'UV', 'margin_vj12']].head(15)

In [ ]:
print("=== Dusty templates, ranked closest-to-boundary first (there are very few) ===")
dusty_ranked = df[df['region'] == 'Dusty'].sort_values('margin_vj12')
dusty_ranked[['name', 'VJ', 'UV', 'margin_vj12']].head(15)

## Check a specific candidate

Type a name from one of the tables above into `candidate_name` and re-run
this cell to see its exact position plotted against the wedge, plus its
margins printed out - a final sanity check before committing to a pick.

In [ ]:
candidate_name = 'NGC_4594'  # <-- change this and re-run

row = df[df['name'] == candidate_name].iloc[0]
print(f"{candidate_name}: VJ={row['VJ']:.3f}  UV={row['UV']:.3f}  region={row['region']}")
print(f"  margin_quiescent = {row['margin_quiescent']:+.3f}  (distance above/below the quiescent lower boundary)")
print(f"  margin_vj12      = {row['margin_vj12']:+.3f}  (distance right/left of the V-J=1.2 divide)")

fig, ax = plt.subplots(figsize=(6, 6))
visualization.plot_uvj_diagram(df['VJ'].values, df['UV'].values, classifications=classifications, ax=ax,
                                title=f'{candidate_name}', scatter_colors=('#dddddd', '#dddddd', '#dddddd'))
ax.scatter([row['VJ']], [row['UV']], color='black', s=120, marker='*', zorder=10, label=candidate_name)
ax.legend(loc='lower right')
plt.show()